 # Pandapower with UK Power Networks


This tutorial shows some functionalities and studies that can be performed using the power flow capabilities of pandapower. It will demonstrate how to run power flow simulations in pandapower, how to analyse the grid and to investigate different use cases relying on the power flow engine of pandapower.

This tutorial has been created in collaboration with UK Power Networks, the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UKPN (LPN, SPN and EPN). It will provide some examples of how pandapower can be used to run investigations and analyses using the open source data released by UK Power Networks. UK Power Networks has provided this grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access:

·       Register and login to [UK Power Networks' Open Data Portal](https://ukpowernetworks.opendatasoft.com/pages/home/)

·       Visit [the LTDS CIM page](https://ukpowernetworks.opendatasoft.com/explore/assets/ukpn-ltds-cim/) and complete [the Shared Data Request Form](https://ukpowernetworks.opendatasoft.com/explore/forms/cim-access-request-form/)



Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

Import the pandapower library and the neccessary methods for the conversion as follows:

In [201]:
import os
import pandapower as pp
from pandapower.converter.cim import from_cim as cim2pp
from pandapower.converter.cim.cim_classes import CimParser
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:,.4f}'.format

## LTDS to pandapower

First, we define the LTDS zip archive, which can be converted to pandapower. If there is no SSH profile available, there is an option to get the generation and load data from LTDS Tables. However, please take into account that there are some assumptions made, which limit the applicability of the data and make them unsuitable for all use cases.

Note: For this tutorial, the network data is imported via the LTDS CIM file, while demand and generation data are derived from LTDS Table 3A – Observed Load and Table 5 – Generation. To enable proper matching of substations in LTDS CIM, these tables have been improved by populating a new MRID column, which provides a unique identification number in CIM. These files are available in the folder where the CIM data was shared.

In [ ]:
# ltds_files is a list containing paths to files needed for the LTDS converter:
ltds_files = "LTDS_LPN_2025_02_EQ_2025-11-28_v1.0.zip"          # Provide the correct directory for the LTDS CIM file
path_excel_demand = "ltds-table-3a-load-data-observed.xlsx"     # Provide the correct directory for the load table
path_excel_generation = "ltds-table-5-generation.xlsx"          # Provide the correct directory for the generation table
excel_column_name = 'Maximum_Demand_24_25_MW'
excel_column_name_pf = 'Maximum_Demand_24_25_PF'
# the Excel data provides the maximum demand / generation. If you want to assume a specific loading,
# choose a scaling_factor between 0.1 and 1.0
scaling_factor = 1.0

cim_parser = CimParser(cgmes_version='ltds')
cim_parser.parse_files(ltds_files).prepare_cim_net().set_cim_data_types()
cim = cim_parser.cim

excel_df_demand = pd.read_excel(path_excel_demand, sheet_name='Feuil1', skiprows=0)
excel_df_generation = pd.read_excel(path_excel_generation, sheet_name='Feuil1', skiprows=0)


In [203]:
def format_uuid_no_dash(uuid_no_dash: str) -> str | None:
    if uuid_no_dash is None:
        return None
    s = str(uuid_no_dash).lstrip("_")
    if len(s) != 32:
        return uuid_no_dash
    parts = [s[0:8], s[8:12], s[12:16], s[16:20], s[20:32]]
    return '_' + '-'.join(parts)
# prepare the demand data
if not excel_df_demand.empty:
    if 'Season' in excel_df_demand:
        excel_df_demand = excel_df_demand.loc[excel_df_demand['Season'] == 'Winter']
    excel_df_demand = excel_df_demand.rename(columns={'Substation MRID': 'name_excel', excel_column_name: 'p_mw_excel', excel_column_name_pf: 'q_mvar_excel'})
    if 'name_excel' not in excel_df_demand or 'p_mw_excel' not in excel_df_demand or 'q_mvar_excel' not in excel_df_demand:
        # the Excel document is not valid
        excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel', 'q_mvar_excel'])
    excel_df_demand = excel_df_demand[['name_excel', 'p_mw_excel', 'q_mvar_excel']].copy()
    excel_df_demand['p_mw_excel'] = excel_df_demand['p_mw_excel'].astype(float)
    excel_df_demand['q_mvar_excel'] = excel_df_demand['q_mvar_excel'].astype(float)
    excel_df_demand['q_mvar_excel'] = ((excel_df_demand['p_mw_excel'] / excel_df_demand['q_mvar_excel'])**2 - excel_df_demand['p_mw_excel']**2)**.5
    excel_df_demand = excel_df_demand.dropna(how='any')
    excel_df_demand_orig = excel_df_demand.copy()
    excel_df_demand = excel_df_demand_orig.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
    excel_df_demand['q_mvar_excel'] = excel_df_demand_orig.groupby('name_excel', as_index=False)['q_mvar_excel'].sum()['q_mvar_excel']
else:
    excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel', 'q_mvar_excel'])
# prepare the generation data
if not excel_df_generation.empty:
    if 'Connected_Accepted' in excel_df_generation:
        excel_df_generation = excel_df_generation.loc[excel_df_generation['Connected_Accepted'] == 'Connected']
    excel_df_generation = excel_df_generation.rename(columns={'Substation MRID': 'name_excel', 'InstalledCapacity_MVA': 'p_mw_excel'})
    if 'name_excel' not in excel_df_generation or 'p_mw_excel' not in excel_df_generation:
        # the Excel document is not valid
        excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])
    excel_df_generation['p_mw_excel'] = excel_df_generation['p_mw_excel'].astype(float)
    excel_df_generation['p_mw_excel'] = excel_df_generation['p_mw_excel'] * -1
    excel_df_generation = excel_df_generation.dropna(how='any')
    # note: there might be an issue with the UUID format, this will be fixed with the following line:
    excel_df_generation['name_excel'] = excel_df_generation['name_excel'].apply(format_uuid_no_dash)

    excel_df_generation = excel_df_generation.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
else:
    excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])

# add the loads to the SSH profile
cim['ssh']['EnergyConsumer'] = pd.concat([cim['ssh']['EnergyConsumer'], cim['eq']['EnergyConsumer'][['rdfId']]], ignore_index=True)
# get the Substation ID
sub = cim['eq']['Terminal'][['ConnectivityNode', 'ConductingEquipment']]
sub = sub.rename(columns={'ConnectivityNode': 'rdfId'})
sub = pd.merge(sub, cim['eq']['ConnectivityNode'][['rdfId', 'ConnectivityNodeContainer']], how='left', on='rdfId')
sub = sub.drop(columns=['rdfId']).drop_duplicates(subset=['ConductingEquipment'])
# adding substations to EnergyConsumer
cim['ssh']['EnergyConsumer']['sub'] = cim['ssh']['EnergyConsumer']['rdfId'].map(
    sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
# identify duplications
cim['ssh']['EnergyConsumer']['dups'] = cim['ssh']['EnergyConsumer'].groupby('sub')['sub'].transform('count')
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['sub'].map(
    excel_df_demand.set_index('name_excel')['p_mw_excel']) / cim['ssh']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['p'].fillna(0.) * scaling_factor
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['sub'].map(
    excel_df_demand.set_index('name_excel')['q_mvar_excel']) / cim['ssh']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['q'].fillna(0.)
cim['ssh']['EnergyConsumer']['inService'] = cim['ssh']['EnergyConsumer']['inService'].fillna(True)
cim['ssh']['EnergyConsumer'] = cim['ssh']['EnergyConsumer'].drop(columns=['sub', 'dups'])

# add the generation to the SSH profile
for one_asset in ['SynchronousMachine', 'PowerElectronicsConnection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    # adding substations to generators
    cim['ssh'][one_asset]['sub'] = cim['ssh'][one_asset]['rdfId'].map(sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
    # identify duplications
    cim['ssh'][one_asset]['dups'] = cim['ssh'][one_asset].groupby('sub')['sub'].transform('count')
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['sub'].map(
        excel_df_generation.set_index('name_excel')['p_mw_excel']) / cim['ssh'][one_asset]['dups']
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)
    cim['ssh'][one_asset] = cim['ssh'][one_asset].drop(columns=['sub', 'dups'])

for one_sw in ['Breaker', 'Disconnector', 'Switch', 'LoadBreakSwitch']:
    cim['ssh'][one_sw] = pd.concat([cim['ssh'][one_sw], cim['eq'][one_sw][['rdfId', 'normalOpen']].rename(columns={'normalOpen': 'open'})], ignore_index=True)
    cim['ssh'][one_sw]['inService'] = True

for one_asset in ['ExternalNetworkInjection', 'ConformLoad', 'NonConformLoad', 'StationSupply',
                  'AsynchronousMachine', 'EquivalentInjection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)

cim['ssh']['ExternalNetworkInjection']['referencePriority'] = cim['ssh']['ExternalNetworkInjection']['referencePriority'].fillna(1)
cim['ssh']['ExternalNetworkInjection']['controlEnabled'] = cim['ssh']['ExternalNetworkInjection']['controlEnabled'].fillna(True)
cim['ssh']['SynchronousMachine']['referencePriority'] = cim['ssh']['SynchronousMachine']['referencePriority'].fillna(0)
cim['ssh']['SynchronousMachine']['controlEnabled'] = cim['ssh']['SynchronousMachine']['controlEnabled'].fillna(False)
cim['ssh']['EquivalentInjection']['regulationStatus'] = cim['ssh']['EquivalentInjection']['regulationStatus'].fillna(False)

cim['ssh']['EnergySource'] = pd.concat([cim['ssh']['EnergySource'], cim['eq']['EnergySource'][['rdfId']]], ignore_index=True)
cim['ssh']['EnergySource']['activePower'] = cim['ssh']['EnergySource']['activePower'].fillna(0.)
cim['ssh']['EnergySource']['reactivePower'] = cim['ssh']['EnergySource']['reactivePower'].fillna(0.)
cim['ssh']['EnergySource']['inService'] = cim['ssh']['EnergySource']['inService'].fillna(True)

cim['ssh']['StaticVarCompensator'] = pd.concat([cim['ssh']['StaticVarCompensator'], cim['eq']['StaticVarCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['StaticVarCompensator']['q'] = cim['ssh']['StaticVarCompensator']['q'].fillna(0.)
cim['ssh']['StaticVarCompensator']['inService'] = cim['ssh']['StaticVarCompensator']['inService'].fillna(True)

# add the terminals
cim['ssh']['Terminal'] = pd.concat([cim['ssh']['Terminal'], cim['eq']['Terminal'][['rdfId']]], ignore_index=True)
cim['ssh']['Terminal']['connected'] = cim['ssh']['Terminal']['connected'].fillna(True)
# add the shunts
cim['ssh']['LinearShuntCompensator'] = pd.concat([cim['ssh']['LinearShuntCompensator'], cim['eq']['LinearShuntCompensator'][['rdfId', 'normalSections']].rename(
    columns={'normalSections': 'sections'})], ignore_index=True)
cim['ssh']['LinearShuntCompensator']['controlEnabled'] = cim['ssh']['LinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['LinearShuntCompensator']['inService'] = cim['ssh']['LinearShuntCompensator']['inService'].fillna(True)
cim['ssh']['NonlinearShuntCompensator'] = pd.concat([cim['ssh']['NonlinearShuntCompensator'], cim['eq']['NonlinearShuntCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['NonlinearShuntCompensator']['sections'] = cim['ssh']['NonlinearShuntCompensator']['sections'].fillna(0)
cim['ssh']['NonlinearShuntCompensator']['controlEnabled'] = cim['ssh']['NonlinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['NonlinearShuntCompensator']['inService'] = cim['ssh']['NonlinearShuntCompensator']['inService'].fillna(True)

# add the tap changer steps
cim['ssh']['RatioTapChanger'] = pd.concat([cim['ssh']['RatioTapChanger'], cim['eq']['RatioTapChanger'][['rdfId', 'neutralStep']].rename(
    columns={'neutralStep': 'step'})], ignore_index=True)
cim['ssh']['RatioTapChanger']['controlEnabled'] = cim['ssh']['RatioTapChanger']['controlEnabled'].fillna(False)
# add the TapChangerControls
cim['ssh']['TapChangerControl'] = pd.concat([cim['ssh']['TapChangerControl'], cim['eq']['TapChangerControl'][['rdfId']]], ignore_index=True)
cim['ssh']['TapChangerControl']['discrete'] = cim['ssh']['TapChangerControl']['discrete'].fillna(False)
cim['ssh']['TapChangerControl']['enabled'] = cim['ssh']['TapChangerControl']['enabled'].fillna(False)
cim['eq']['PowerTransformer']['inService'] = True

cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['EquivalentBranch'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['ACLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['DCLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)

# use the from_cim_dict to put in the modified CimParser
net = cim2pp.from_cim_dict(cim_parser=cim_parser, cim_version='LTDS', create_tap_controller=False)

# if there is no slack in the grid, create one
if net.gen.empty and net.ext_grid.empty and not net.sgen.empty:
    slack = net.sgen.loc[net.sgen.in_service].loc[net.sgen.p_mw == net.sgen.p_mw.max()]
    net.sgen = net.sgen.drop(slack.index[0])
    pp.create_gen(net, bus=slack.bus.iloc[0], p_mw=slack.p_mw.iloc[0], slack=True, in_service=True)

print('Conversion successful')

Conversion successful


## Get an overview over your grid
Once the network is converted to pandapower, the data can be displayed:

In [204]:
print(net)


This pandapower network includes the following parameter tables:
   - bus (10746 elements)
   - load (2892 elements)
   - sgen (7 elements)
   - gen (1 element)
   - switch (13289 elements)
   - shunt (14 elements)
   - line (944 elements)
   - trafo (485 elements)
   - trafo3w (97 elements)
   - impedance (100 elements)
   - ward (77 elements)


## Export your grid
There are different options to export, for example as JSON, Excel or CSV:

In [ ]:
path_to_json = "UKPN_grid.json"
path_to_excel = "UKPN_grid.xlsx"
pp.to_json(net, path_to_json)
pp.to_excel(net, path_to_excel)

Could not serialize net.CGMES
Could not serialize net.report_container


In [ ]:
net.line.to_excel("line_data.xlsx")
net.trafo.to_excel("trafo_data.xlsx")
net.bus.to_excel("bus_data.xlsx")

## Display the nodes

In [207]:
print(net.bus.iloc[0])

name                                                             T_39
vn_kv                                                        132.0000
type                                                                n
zone                                                    Wimbledon 3&4
in_service                                                       True
geo                                                               NaN
origin_id                       _ef8b046a-870d-4c83-9071-da2f0ebec399
origin_class                                         ConnectivityNode
origin_profile                                                     eq
cim_topnode                                                       NaN
ConnectivityNodeContainer_id    _d3a9e549-ea05-4ab2-8e5a-d45ee4ad53ca
Substation_id                   _d2d3ea2c-a4e0-4d3b-9fb6-aae460f2ef3a
description                                                       NaN
Busbar_id                                                         NaN
Busbar_name         

## Display the lines

In [208]:
print(net.line.iloc[0])

name                                           lne_BRWB_WFCS_1
std_type                                                  None
from_bus                                                  6160
to_bus                                                    7589
length_km                                               2.1750
r_ohm_per_km                                            0.1290
x_ohm_per_km                                            0.1260
c_nf_per_km                                           142.0009
g_us_per_km                                             0.0000
max_i_ka                                                   NaN
df                                                      1.0000
parallel                                                     1
type                                                      None
in_service                                                True
geo                                                        NaN
origin_id                _00122e74-780a-493b-904b-37dff

## Display the transformers

In [209]:
# two winding transformers
print(net.trafo.iloc[0])

name                                                                             T4
std_type                                                                       None
hv_bus                                                                         7874
lv_bus                                                                         8511
sn_mva                                                                      10.0000
vn_hv_kv                                                                    22.0000
vn_lv_kv                                                                     6.6000
vk_percent                                                                  11.2956
vkr_percent                                                                  0.7600
pfe_kw                                                                       0.0000
i0_percent                                                                   0.0000
shift_degree                                                                

In [210]:
# three winding transformers
print(net.trafo3w.iloc[0])

name                                                                            GT1
std_type                                                                       None
hv_bus                                                                         9001
mv_bus                                                                        10064
lv_bus                                                                         2409
sn_hv_mva                                                                   60.6000
sn_mv_mva                                                                   30.3000
sn_lv_mva                                                                   30.3000
vn_hv_kv                                                                   132.0000
vn_mv_kv                                                                    11.1100
vn_lv_kv                                                                    11.1100
vk_hv_percent                                                               

## Display the loads

In [211]:
print(net.load.iloc[0])

name                                                    W2
bus                                                   7547
p_mw                                                1.8105
q_mvar                                              0.4538
const_z_p_percent                                   0.0000
const_i_p_percent                                   0.0000
const_z_q_percent                                   0.0000
const_i_q_percent                                   0.0000
sn_mva                                                 NaN
scaling                                             1.0000
in_service                                            True
type                                                  None
origin_id            _001042be-8d96-4f41-bcea-c835cdac77fb
origin_class                                EnergyConsumer
terminal             _172fc73f-33ec-9899-2c3b-66203a28cc94
description                                            NaN
Name: 0, dtype: object


## Display the generation

In [212]:
print(net.sgen.iloc[0])

name                                                       LPN 0007(27)
bus                                                                4495
p_mw                                                            -0.0000
q_mvar                                                          -0.0000
min_q_mvar                                                          NaN
max_q_mvar                                                          NaN
sn_mva                                                              NaN
scaling                                                          1.0000
controllable                                                      False
id_q_capability_characteristic                                     <NA>
reactive_capability_curve                                           NaN
curve_style                                                         NaN
in_service                                                         True
type                                                     Generat

## Get only HV elements from the grid
In pandapower we are using pandas DataFrames, you can create your queries like you wish. Here is an example to get HV (110kV) elements from your grid.

In [213]:
print("the HV nodes first")
print(net.bus.loc[(net.bus.vn_kv > 100) & (net.bus.vn_kv < 150)])

the HV nodes first
                   name    vn_kv type                zone  in_service  geo  \
0                  T_39 132.0000    n       Wimbledon 3&4        True  NaN   
16      CITY11 M BB1(2) 132.0000    n           City Road        True  NaN   
45      DEPT11 BB3_2(2) 132.0000    n       Deptford Grid        True  NaN   
54            HURS12(1) 132.0000    n               Hurst        True  NaN   
58            HURS11(1) 132.0000    n               Hurst        True  NaN   
76          Terminal(7) 132.0000    n            Bankside        True  NaN   
78                  T_5 132.0000    n      Kingston (SPN)        True  NaN   
79         HASC11 R BB2 132.0000    b   Hackney Supergrid        True  NaN   
81               T_8(1) 132.0000    n   Brunswick Wharf B        True  NaN   
96               EBBR12 132.0000    n        Ebury Bridge        True  NaN   
104           HOLL11(2) 132.0000    n            Holloway        True  NaN   
105     NEWC11 M BB1(5) 132.0000    n     New

In [214]:
print("now the HV lines")
net.line['vn_kv_bus'] = net.line.from_bus.map(net.bus.vn_kv)
print(net.line.loc[(net.line.vn_kv_bus > 100) & (net.line.vn_kv_bus < 150)])

now the HV lines
                                   name std_type  from_bus  to_bus  length_km  \
0                       lne_BRWB_WFCS_1     None      6160    7589     2.1750   
3                       lne_NEWC_TOOL_1     None     10443    9394     4.4000   
6                       lne_WISD_FULC_1     None      4538    2232    12.1200   
11                      lne_WM12_BENG_2     None      1045    3982     9.1400   
15                      Lne_BANF_FISB_3     None      6060    9578     3.0000   
18                      lne_LITB_DART_3     None     10535    2343     3.7600   
20                    lne_ELTH_ELTM_T1B     None      8760    4450     1.0000   
32                      lne_HURS_BROM_1     None      9043      54     1.0000   
35                     lne_ISLNG_GEORG1     None      6661    3988     2.8000   
42                      lne_CITY_CITB_2     None      3821    2912     1.0000   
45                       L_BANK_BANDGT5     None      6067    2815     1.0000   
47         

In [215]:
print("now the HV 2W trafos")
print(net.trafo.loc[(net.trafo.vn_hv_kv > 100) & (net.trafo.vn_hv_kv < 150)])

now the HV 2W trafos
            name std_type  hv_bus  lv_bus   sn_mva  vn_hv_kv  vn_lv_kv  \
1            GT1     None    5102    8079  60.0000  132.0000   33.0000   
4            GT3     None   10409     929  60.0000  132.0000   22.0000   
8        T1_EECL     None    2752     715 520.0000  132.0000   22.0000   
15          GT1C     None    1282    6043  90.0000  132.0000   33.0000   
19          GT4B     None    2186    6044  15.0000  132.0000   11.0000   
21          GT1A     None   10439    6381  15.0000  132.0000   11.0000   
26          GT2B     None    8333    8571  90.0000  132.0000   33.0000   
44          GT2A     None    7194    4217  15.0000  132.0000   11.0000   
51          GT1A     None    9854    5467  15.0000  132.0000   11.0000   
58           GT1     None    6107    1722  60.0000  132.0000   33.0000   
59          GT3B     None    6883    9395  15.0000  132.0000   11.0000   
62          GT2B     None    7252    6358  90.0000  132.0000   66.0000   
63   LPN 0045(3) 

In [216]:
print("now the HV 3W trafos")
print(net.trafo3w.loc[(net.trafo3w.vn_hv_kv > 100) & (net.trafo3w.vn_hv_kv < 150)])

now the HV 3W trafos
           name std_type  hv_bus  mv_bus  lv_bus  sn_hv_mva  sn_mv_mva  \
0           GT1     None    9001   10064    2409    60.6000    30.3000   
1           GT3     None    3512    7750   10142    66.6000    33.3000   
2           GT2     None    4179     278    9035    60.6000    30.3000   
4            T2     None    1223     631    8844    60.6000    30.3000   
5           GT2     None    2889    8029    5052    66.6000    33.3000   
6           GT2     None    3028      49   10037    60.6000    30.3000   
7           GT2     None    5922    3176    5989    66.6000    33.3000   
8   LPN 0044(4)     None    8175    2075    7664    85.0000    42.5000   
9   LPN 0006(2)     None    7105    2297    3834    60.6000    30.3000   
10          GT5     None    2815     710    2141    90.0000    45.0000   
11         GT1A     None    4457    7675    8884    66.6600    33.3300   
12          GT1     None    5622    9282    4725    66.6600    33.3300   
13          GT2  

## Get MV Loads and Generation

In [217]:
print("now the loads")
net.load['vn_kv_bus'] = net.load.bus.map(net.bus.vn_kv)
print(net.load.loc[(net.load.vn_kv_bus == 11)].head(10))

now the loads
   name   bus   p_mw  q_mvar  const_z_p_percent  const_i_p_percent  \
0    W2  7547 1.8105  0.4538             0.0000             0.0000   
1    E1  9988 1.6625  0.3376             0.0000             0.0000   
3    E1  1749 2.4200  0.4914             0.0000             0.0000   
4   SE2  6411 2.2200  0.6475             0.0000             0.0000   
5   WH5  8445 1.2659  0.4595             0.0000             0.0000   
6   MP1   178 1.8765  0.4703             0.0000             0.0000   
7    B1  2375 1.5333  0.5040             0.0000             0.0000   
8   SD3   188 0.2882  0.0841             0.0000             0.0000   
9   NE2  8775 1.8625  0.4668             0.0000             0.0000   
10  WA2  9529 0.1750  0.0510             0.0000             0.0000   

    const_z_q_percent  const_i_q_percent  sn_mva  scaling  in_service  type  \
0              0.0000             0.0000     NaN   1.0000        True  None   
1              0.0000             0.0000     NaN   1.0000

In [218]:
print("now the PQ generators")
net.sgen['vn_kv_bus'] = net.sgen.bus.map(net.bus.vn_kv)
print(net.sgen.loc[(net.sgen.vn_kv_bus == 11)].head(10))

now the PQ generators
                       name   bus    p_mw  q_mvar  min_q_mvar  max_q_mvar  \
2  REI_B12_PQ__Plus__030deg   700 -0.0000 -0.0000         NaN         NaN   
3   REI_B3_PV__Plus__059deg  9442 -0.0000 -0.0000         NaN         NaN   
6   REI_B3_PQ__Plus__030deg  6431 -0.0000 -0.0000         NaN         NaN   
7  REI_B3_PQ__Minus__030deg  2330 -0.0000 -0.0000         NaN         NaN   

   sn_mva  scaling  controllable  id_q_capability_characteristic  ... vn_kv  \
2     NaN   1.0000         False                            <NA>  ...   NaN   
3     NaN   1.0000         False                            <NA>  ...   NaN   
6     NaN   1.0000         False                            <NA>  ...   NaN   
7     NaN   1.0000         False                            <NA>  ...   NaN   

  rdss_ohm  xdss_pu lrc_pu  RegulatingControl.targetValue referencePriority  \
2      NaN      NaN    NaN                            NaN            0.0000   
3      NaN      NaN    NaN            

# Run a power flow

#### Pre-processing steps for power flow execution
The following blocks of code provide functions to apply the adjustments necessary to successfully run the power flow on the UK Power Networks grids. These adjustments are required due to differences in how the pandapower library handles network data compared with other power system software tools, such as PowerFactory. Examples include the creation of external grids (*slack buses* in power flow terminology) and the replacement of zero-impedance components with switches.

In [219]:
# Function to replace components with very small impedance with switches.
from pandapower.toolbox import create_replacement_switch_for_branch

def _replace_zero_impedance_components(net):
    min_ohm = 0.001
    to_replace = (np.abs(net.line.x_ohm_per_km * net.line.length_km) <= min_ohm) & net.line.in_service

    if np.any(to_replace):
        print(f"replaced {sum(to_replace)} lines with switches")

    for i in net.line.loc[to_replace].index.values:
        create_replacement_switch_for_branch(net, "line", i)
        net.line.at[i, "in_service"] = False

    xward = net.xward.loc[(np.abs(net.xward.x_ohm) <= min_ohm) & net.xward.in_service].index.values
    if len(xward) > 0:
        pp.replace_xward_by_ward(net, index=xward, drop=False)
        print(f"replaced {len(xward)} xwards with wards")

    zb_f_ohm = np.square(net.bus.loc[net.impedance.from_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    zb_t_ohm = np.square(net.bus.loc[net.impedance.to_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    impedance = ((np.abs(net.impedance.xft_pu) <= min_ohm / zb_f_ohm) |
                (np.abs(net.impedance.xtf_pu) <= min_ohm / zb_t_ohm)) & net.impedance.in_service

    if any(impedance):
        print(f"replaced {sum(impedance)} impedance elements with switches")

    for i in net.impedance.loc[impedance].index.values:
        pp.create_replacement_switch_for_branch(net, "impedance", i)
        net.impedance.at[i, "in_service"] = False

In [220]:
# Function to apply the needed preprocessing steps
def apply_preprocessing(net, license_area):
    # Here zero impedance components are  modified to switches to prevent ill-conditioning
    net.impedance.drop(net.impedance.index, inplace=True)
    net.line["c_nf_per_km"] *= 0.1
    _replace_zero_impedance_components(net)
    
    # Here powers are scaled down, as Excel tables provided max demand values. 
    # When using SSH profiles, it will be possible to avoid this.
    net.load["p_mw"] *= 0.1
    net.load["q_mvar"] *= 0.1
    net.sgen["p_mw"] *= 0.5
    net.sgen["q_mvar"] *= 0.5

    # Here slack buses are defined for each grid. 
    # Slack buses will be defined in the SSH profiles, so this steps won't be needed 
    # when using the full CIM data with SSH profiles.
    if license_area == "LPN":
        pp.create_ext_grid(net,bus=10711,vm_pu=1)
        pp.create_ext_grid(net,bus=10699,vm_pu=1)
        pp.create_ext_grid(net,bus=10674,vm_pu=1)
        pp.create_ext_grid(net,bus=10738,vm_pu=1)
        pp.create_ext_grid(net,bus=10673,vm_pu=1)
    elif license_area == "SPN":
        pp.create_ext_grid(net,bus=4899,vm_pu=1)
        pp.create_ext_grid(net,bus=4879,vm_pu=1)
        pp.create_ext_grid(net,bus=4903,vm_pu=1)
        pp.create_ext_grid(net,bus=4920,vm_pu=1)
        pp.create_ext_grid(net,bus=4916,vm_pu=1)
        pp.create_ext_grid(net,bus=4925,vm_pu=1)
        pp.create_ext_grid(net,bus=4878,vm_pu=1)
    elif license_area == "EPN":
        pp.create_ext_grid(net,bus=9906,vm_pu=1)
        pp.create_ext_grid(net,bus=9918,vm_pu=1)
        pp.create_ext_grid(net,bus=9900,vm_pu=1)
        pp.create_ext_grid(net,bus=9910,vm_pu=1)
        pp.create_ext_grid(net,bus=9878,vm_pu=1)
    else:
        raise ValueError("Sorry, this license area does not exist in UK Power Networks. Allowed areas are LPN, SPN and EPN.")

    return net

In [221]:
# Apply the pre-processing steps on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    net = apply_preprocessing(net, license_area)

replaced 154 lines with switches


## Run a power flow study
One of the easiest tasks that can be done with pandapower is to run a power flow. 
This allows analysing the voltage conditions in the grid and the powers/currents flowing through the different lines and components of the network, considering the load and generation available as input. 

Through a power flow calculation it is possible to make a contingency analysis, namely to assess if the operating conditions of the grid are within the allowed boundaries.

In this section, you will see: 
- How to run a power flow and visualize the results
- How to filter the power flow results
- How to identify possible contingencies (overloading or voltage violations)



In [222]:
# Run a power flow
if len(net.bus) == 0:
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)
pp.runpp(net, max_iteration=50)

## Visualize the power flow results
In the bus results table you will find the resulting bus voltage and power consumption / injection at each bus

In [223]:
# Visualize bus results
display(net.res_bus.loc[:5])
display("Maximum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)))
display("Minimum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)))

,vm_pu,va_degree,p_mw,q_mvar
0,1.0000,0.0000,0.0000,0.0000
1,0.9781,-5.6045,0.1819,0.0456
2,0.9423,-11.4989,0.4500,0.1313
3,0.9943,-1.1768,0.0000,0.0000
4,0.9762,-6.0355,0.2230,0.0650
5,0.9872,-11.4144,0.0000,0.0000


'Maximum voltage magnitude in the grid (per unit): 1.0492'

'Minimum voltage magnitude in the grid (per unit): 0.8657'

Some of the bus results may have NaN. This happens for those buses that are disconnected from the main grid.

In [224]:
# Visualize number of connected buses
num_disconnected_buses = np.sum(np.isnan(net.res_bus.vm_pu))
num_connected_buses = np.sum(~np.isnan(net.res_bus.vm_pu))
total_num_buses = len(net.bus)
percentage_connected_buses = 100 * num_connected_buses / total_num_buses
display("Percentage of connected buses: " + "{:.2f}".format(percentage_connected_buses) + "%")

'Percentage of connected buses: 92.29%'

In the line and transformer result tables you can see, among others, the level of power flowing through these components.

In [225]:
# Visualize line results
display(net.res_trafo.loc[:5])

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.4684,0.1565,-0.4682,-0.1537,0.0002,0.0028,0.0131,0.0437,0.9891,-17.4632,0.9869,-17.7661,4.9932
1,1.8893,0.2624,-1.8460,-0.1564,0.0432,0.1061,0.0084,0.0325,0.9979,-0.1122,0.9972,-0.3770,3.1858
2,0.7370,0.1915,-0.7368,-0.1845,0.0003,0.0070,0.0134,0.0401,0.9962,-0.4892,0.9937,-0.9954,5.0959
3,-3.2927,-0.8838,3.3091,1.1417,0.0163,0.2579,0.0683,0.2019,0.8739,-17.5752,0.9099,-15.0219,26.2892
4,1.9671,0.5275,-1.9228,-0.4115,0.0443,0.1160,0.0091,0.0530,0.9752,-3.9193,0.9739,-4.1673,3.4807
5,0.8642,0.2177,-0.8638,-0.2083,0.0004,0.0094,0.0161,0.0484,0.9667,-4.5656,0.9638,-5.1460,6.1459


In [226]:
# Visualize transformer results
display(net.res_trafo.loc[:5])

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.4684,0.1565,-0.4682,-0.1537,0.0002,0.0028,0.0131,0.0437,0.9891,-17.4632,0.9869,-17.7661,4.9932
1,1.8893,0.2624,-1.8460,-0.1564,0.0432,0.1061,0.0084,0.0325,0.9979,-0.1122,0.9972,-0.3770,3.1858
2,0.7370,0.1915,-0.7368,-0.1845,0.0003,0.0070,0.0134,0.0401,0.9962,-0.4892,0.9937,-0.9954,5.0959
3,-3.2927,-0.8838,3.3091,1.1417,0.0163,0.2579,0.0683,0.2019,0.8739,-17.5752,0.9099,-15.0219,26.2892
4,1.9671,0.5275,-1.9228,-0.4115,0.0443,0.1160,0.0091,0.0530,0.9752,-3.9193,0.9739,-4.1673,3.4807
5,0.8642,0.2177,-0.8638,-0.2083,0.0004,0.0094,0.0161,0.0484,0.9667,-4.5656,0.9638,-5.1460,6.1459


You can visualize the results for a specific element

In [227]:
# Visualize bus results at a selected bus
bus_idx = 178
if bus_idx in net.res_bus.index:
    print(net.res_bus.loc[bus_idx])
else:
    print("The given bus does not exist")

vm_pu        0.9700
va_degree   -4.9735
p_mw         0.1876
q_mvar       0.0470
Name: 178, dtype: float64


In [228]:
# Visualize results at a selected transformer
trafo_idx = 15
if trafo_idx in net.res_trafo.index:
    print(net.res_trafo.loc[trafo_idx])
else:
    print("The given transformer does not exist")

p_hv_mw            0.8523
q_hv_mvar         -0.2747
p_lv_mw           -0.8522
q_lv_mvar          0.3094
pl_mw              0.0000
ql_mvar            0.0347
i_hv_ka            0.0040
i_lv_ka            0.0163
vm_hv_pu           0.9740
va_hv_degree      -4.4167
vm_lv_pu           0.9747
va_lv_degree      -4.5499
loading_percent    1.0335
Name: 15, dtype: float64


## Sort the results 
You can easily sort the results using the *sort_values* function


In [229]:
# Sort bus results from buses with the smallest voltage
net.res_bus.sort_values("vm_pu").head(5)

,vm_pu,va_degree,p_mw,q_mvar
2471,0.8657,-17.8511,0.0000,0.0000
6098,0.8657,-17.8530,0.0000,0.0000
7634,0.8658,-17.8435,0.0000,0.0000
4460,0.8659,-17.8452,0.0000,0.0000
6463,0.8659,-17.8533,0.0000,0.0000


In [230]:
# Sort line results from lines with highest active power flow
net.res_line.sort_values("p_from_mw", ascending=False).head(5)

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
538,134.4007,-5.2952,-133.8342,5.8453,0.5665,0.5501,0.6009,0.6008,0.6009,0.9790,-3.4636,0.9752,-3.9193,NaN
708,132.2813,-8.9497,-131.8767,9.3321,0.4046,0.3825,0.5925,0.5924,0.5925,0.9787,-3.4619,0.9761,-3.7962,NaN
617,126.8456,-9.6059,-126.7513,9.5901,0.0942,-0.0158,0.5706,0.5704,0.5706,0.9752,-3.9193,0.9747,-4.1113,NaN
389,126.8456,-9.6059,-126.7513,9.5901,0.0942,-0.0158,0.5706,0.5704,0.5706,0.9752,-3.9193,0.9747,-4.1113,NaN
380,124.8627,-8.9600,-124.8083,8.3419,0.0544,-0.6181,0.5617,0.5615,0.5617,0.9747,-4.1113,0.9744,-4.2323,NaN


## Filter the results 
You can filter the results as you like, selecting only specific types or clusters of elements, or specific columns of the tables

In [231]:
# Visualize bus results only for buses at 132 kV
net.res_bus[net.bus.vn_kv==132].head(5)

,vm_pu,va_degree,p_mw,q_mvar
0,1.0000,0.0000,0.0000,0.0000
16,0.9740,-4.4157,0.0000,0.0000
45,0.9752,-3.9193,0.0000,0.0000
54,0.9980,-0.1132,0.0000,0.0000
58,0.9979,-0.0951,0.0000,0.0000


In [232]:
# Visualize transformer results only for 132 kV/33 kV  transformers 
net.res_trafo[(net.trafo.vn_hv_kv==132) & (net.trafo.vn_lv_kv==33)].head(5)

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
1,1.8893,0.2624,-1.8460,-0.1564,0.0432,0.1061,0.0084,0.0325,0.9979,-0.1122,0.9972,-0.3770,3.1858
15,0.8523,-0.2747,-0.8522,0.3094,0.0000,0.0347,0.0040,0.0163,0.9740,-4.4167,0.9747,-4.5499,1.0335
26,6.2574,1.4172,-6.1979,-1.0878,0.0595,0.3294,0.0281,0.1105,1.0000,-0.0004,0.9964,-0.9005,7.1287
58,0.0160,0.0670,0.0000,0.0000,0.0160,0.0670,0.0003,0.0000,0.9740,-4.4157,0.9739,-4.4170,0.1178
63,-0.0000,0.0000,0.0000,-0.0000,0.0000,0.0000,0.0000,0.0000,0.9823,-2.7178,0.9823,-2.7178,0.0000


In [233]:
# Visualize only active and reactive powers of the lines
net.res_line[["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar"]].head(5)

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar
0,1.8853,0.4704,-1.8852,-0.6192
1,0.0006,-0.0762,-0.0006,0.0134
2,0.0000,0.0000,0.0000,0.0000
3,124.8627,-8.9600,-124.8083,8.3419
4,0.8931,0.4482,-0.8929,-0.7239


## Detect contingencies
You can easily identify possible voltage contingencies in the grid, namely voltage values beyond the allowed thresholds. 

In [234]:
# Check possible voltage violations
# Define voltage boundaries
lower_v_threshold = 0.90   # Define the lower boundary of the voltage magnitude (in per unit)
upper_v_threshold = 1.10   # Define the upper boundary of the voltage magnitude (in per unit)

# Check for overvoltages
if np.any(net.res_bus.vm_pu > upper_v_threshold):
    display("Overvoltages are present in the grid. Maximum voltage is: " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)) + " p.u.")
    buses_with_overvoltage = net.bus.index[net.res_bus.vm_pu>upper_v_threshold]
    print("Buses with overvoltage:")
    print(buses_with_overvoltage)
else: 
    display("No overvoltages are present in the grid")

# Check for undervoltages
if np.any(net.res_bus.vm_pu < lower_v_threshold):
    display("Undervoltages are present in the grid. Minimum voltage is: " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)) + " p.u.")
    buses_with_undervoltage = net.bus.index[net.res_bus.vm_pu<lower_v_threshold]
    print("Buses with undervoltage:")
    print(buses_with_undervoltage)
else: 
    display("No undervoltages are present in the grid")



'No overvoltages are present in the grid'

'Undervoltages are present in the grid. Minimum voltage is: 0.8657 p.u.'

Buses with undervoltage:
Index([   12,    42,    97,   102,   134,   153,   174,   230,   307,   322,
       ...
       10578, 10582, 10591, 10639, 10678, 10687, 10693, 10720, 10726, 10739],
      dtype='int64', length=256)


You can easily check if any overloading exists in the grid (**NB**: the possibility of verifying overloadings depends on the availability of rated values for lines and/or transformers).

In [235]:
# Check overloadings for transformers

overloading_factor = 1  # You can define an overloading factor if you desire to check overloadings for values different from 100%
if np.any(net.res_trafo.loading_percent > 100*overloading_factor):
    display("Overloading present in the grid transformers. Maximum loading is: " + "{:.2f}".format(np.nanmax(net.res_trafo.loading_percent)) + "%")
else: 
    display("No overloading is present in the grid transformers. Maximum loading is: " + "{:.2f}".format(np.nanmax(net.res_trafo.loading_percent)) + "%")

'Overloading present in the grid transformers. Maximum loading is: 108.94%'

In [236]:
# Visualize transformer loading (results sorted by the largest loading)
net.res_trafo[["loading_percent"]].sort_values("loading_percent", ascending=False).head(5)

,loading_percent
24,108.9383
164,59.6218
310,59.6178
399,33.4610
3,26.2892


# Use Case Study: Headroom capacity
#### Impact of new load or generation connections

Headroom capacity studies are a common use case that can be addressed leveraging the pandapower power flow libraries. The goal is to understand how much load or generation can be connected to a bus, before exceeding the allowed boundaries (voltage boundaries or overloading of the grid components).

In this section you will see:
- How to add new loads or generators to the grid
- How to discover the maximum load or generation that can be added at a bus before exceeding the operational boundaries (i.e., voltage or overloading limits)

"In this use case study, a new load and generation connection to the Barking West 11kV busbar is analysed. The single line diagram is available on [the LTDS landing page](https://ukpowernetworks.opendatasoft.com/explore/assets/long-term-development-statement/view/) on UK Power Networks' Open Data Portal.

#### 1. Derive the data for the busbar "BARW52 BB1"


In [237]:
# Get the index of the bus with the name "BARW52 BB1"
BARW52_index = net.bus[net.bus["name"] == "BARW52 BB1"].index[0]
print("The index of searched bus is: " + str(BARW52_index))

The index of searched bus is: 8287


In [238]:
# Get the bus results before adding load to analyse their loading after the load is added
before_BARW52 = net.res_bus.loc[BARW52_index]
print(before_BARW52)

vm_pu         0.9038
va_degree   -18.1756
p_mw          0.0000
q_mvar        0.0000
Name: 8287, dtype: float64


### 2. Derive the data for the transformers "FT2" and "T3" connected to the busbar "BARW52 BB1".

In [239]:
# Get the indices of the nodes with the names "BARW3B" and "BARW3C" where trafos are connected to them
bus_ft2 = net.bus[net.bus["name"] == "BARW3B"].index[0]
bus_t3 = net.bus[net.bus["name"] == "BARW3C"].index[0]

print("BARW3B node index:", bus_ft2)
print("BARW3C node index:", bus_t3)

# Find the transformers connected to these nodes
trafo_ft2 = net.trafo[net.trafo["hv_bus"] == bus_ft2].index[0]
trafo_t3 = net.trafo[net.trafo["hv_bus"] == bus_t3].index[0]

print("FT2 trafo index:", trafo_ft2, "| name:", net.trafo.at[trafo_ft2, "name"])
print("T3 trafo index:", trafo_t3, "| name:", net.trafo.at[trafo_t3, "name"])

BARW3B node index: 4723
BARW3C node index: 10214
FT2 trafo index: 197 | name: FT2
T3 trafo index: 360 | name: T3


In [240]:
# Get the transformer results before adding load to analyse their loading after the load is added
before_ft2 = net.res_trafo.loc[trafo_ft2].copy()
before_t3 = net.res_trafo.loc[trafo_t3].copy()

print("Before adding load:")
print("FT2:\n", before_ft2)
print("T3:\n", before_t3)

Before adding load:
FT2:
 p_hv_mw             0.5906
q_hv_mvar           0.1803
p_lv_mw            -0.5833
q_lv_mvar          -0.1462
pl_mw               0.0073
ql_mvar             0.0341
i_hv_ka             0.0125
i_lv_ka             0.0349
vm_hv_pu            0.8663
va_hv_degree      -17.8374
vm_lv_pu            0.9038
va_lv_degree      -18.1756
loading_percent     3.9605
Name: 197, dtype: float64
T3:
 p_hv_mw             0.5906
q_hv_mvar           0.1803
p_lv_mw            -0.5833
q_lv_mvar          -0.1462
pl_mw               0.0073
ql_mvar             0.0341
i_hv_ka             0.0125
i_lv_ka             0.0349
vm_hv_pu            0.8663
va_hv_degree      -17.8374
vm_lv_pu            0.9038
va_lv_degree      -18.1756
loading_percent     3.9605
Name: 360, dtype: float64


### 3. Derive the data for the lines connected to the transformers "FT2" and "T3".


In [241]:
# Find and print lines connected to the transformers FT2 and T3 
lines_to_ft2 = net.line[net.line["to_bus"] == bus_ft2]
lines_to_t3 = net.line[net.line["to_bus"] == bus_t3]

print("Lines connected to FT2 (BARW3B):\n", lines_to_ft2[["name", "from_bus", "to_bus"]])
print("Lines connected to T3 (BARW3C):\n", lines_to_t3[["name", "from_bus", "to_bus"]])

Lines connected to FT2 (BARW3B):
                 name  from_bus  to_bus
343  lne_BARW_BARD_2      4795    4723
Lines connected to T3 (BARW3C):
                 name  from_bus  to_bus
358  lne_BARW_BARD_3      9764   10214


In [242]:
# Get and print the results for lines connected to the transformers FT2 and T3 
# before adding the new load

before_lines_ft2 = net.res_line.loc[lines_to_ft2.index]
before_lines_t3 = net.res_line.loc[lines_to_t3.index]

print("Results for lines at FT2:\n", before_lines_ft2)
print("Results for lines at T3:\n", before_lines_t3)

Results for lines at FT2:
      p_from_mw  q_from_mvar  p_to_mw  q_to_mvar  pl_mw  ql_mvar  i_from_ka  \
343     0.5906       0.1797  -0.5906    -0.1803 0.0000  -0.0006     0.0125   

     i_to_ka   i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
343   0.0125 0.0125      0.8663        -17.8369    0.8663      -17.8374   

     loading_percent  
343              NaN  
Results for lines at T3:
      p_from_mw  q_from_mvar  p_to_mw  q_to_mvar  pl_mw  ql_mvar  i_from_ka  \
358     0.5906       0.1797  -0.5906    -0.1803 0.0000  -0.0006     0.0125   

     i_to_ka   i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
358   0.0125 0.0125      0.8663        -17.8369    0.8663      -17.8374   

     loading_percent  
358              NaN  


## Add a new load 
A new load can be easily created with the *create_load" function of pandapower. It requires defining the bus to which the load will be connected and its active and reactive power.


In [243]:
# Create a new load at the desired bus
load_bus = BARW52_index

if load_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=load_bus)
load_p = 1 #MW
load_q = 0 #MVAr
pp.create_load(net, bus=load_bus, p_mw=load_p, q_mvar=load_q)
net.load.tail(1)

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type,origin_id,origin_class,terminal,description,vn_kv_bus
2892,None,8287,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,1.0000,True,wye,NaN,NaN,NaN,NaN,NaN


In [244]:
# Run a power flow after adding the new load
pp.runpp(net, max_iteration=50)

In [245]:
# Get the bus results after adding load to analyse their loading after the load is added
after_BARW52 = net.res_bus.loc[BARW52_index]
print(after_BARW52)

vm_pu         0.8960
va_degree   -19.6192
p_mw          1.0000
q_mvar        0.0000
Name: 8287, dtype: float64


In [246]:
# Get the transformer results after adding load to analyse their loading after the load is added
after_ft2 = net.res_trafo.loc[trafo_ft2].copy()
after_t3 = net.res_trafo.loc[trafo_t3].copy()

print("After adding load:")
print("FT2:\n", after_ft2)
print("T3:\n", after_t3)

After adding load:
FT2:
 p_hv_mw             0.9242
q_hv_mvar           0.1851
p_lv_mw            -0.9167
q_lv_mvar          -0.1462
pl_mw               0.0075
ql_mvar             0.0389
i_hv_ka             0.0192
i_lv_ka             0.0544
vm_hv_pu            0.8590
va_hv_degree      -19.0765
vm_lv_pu            0.8960
va_lv_degree      -19.6192
loading_percent     6.0957
Name: 197, dtype: float64
T3:
 p_hv_mw             0.9242
q_hv_mvar           0.1851
p_lv_mw            -0.9167
q_lv_mvar          -0.1462
pl_mw               0.0075
ql_mvar             0.0389
i_hv_ka             0.0192
i_lv_ka             0.0544
vm_hv_pu            0.8590
va_hv_degree      -19.0765
vm_lv_pu            0.8960
va_lv_degree      -19.6192
loading_percent     6.0957
Name: 360, dtype: float64


In [247]:
comparison_ft2 = pd.DataFrame({"before": before_ft2, "after": after_ft2})
comparison_t3 = pd.DataFrame({"before": before_t3, "after": after_t3})

print("FT2 comparison:\n", comparison_ft2)
print("T3 comparison:\n", comparison_t3)

FT2 comparison:
                   before    after
p_hv_mw           0.5906   0.9242
q_hv_mvar         0.1803   0.1851
p_lv_mw          -0.5833  -0.9167
q_lv_mvar        -0.1462  -0.1462
pl_mw             0.0073   0.0075
ql_mvar           0.0341   0.0389
i_hv_ka           0.0125   0.0192
i_lv_ka           0.0349   0.0544
vm_hv_pu          0.8663   0.8590
va_hv_degree    -17.8374 -19.0765
vm_lv_pu          0.9038   0.8960
va_lv_degree    -18.1756 -19.6192
loading_percent   3.9605   6.0957
T3 comparison:
                   before    after
p_hv_mw           0.5906   0.9242
q_hv_mvar         0.1803   0.1851
p_lv_mw          -0.5833  -0.9167
q_lv_mvar        -0.1462  -0.1462
pl_mw             0.0073   0.0075
ql_mvar           0.0341   0.0389
i_hv_ka           0.0125   0.0192
i_lv_ka           0.0349   0.0544
vm_hv_pu          0.8663   0.8590
va_hv_degree    -17.8374 -19.0765
vm_lv_pu          0.9038   0.8960
va_lv_degree    -18.1756 -19.6192
loading_percent   3.9605   6.0957


In [248]:
# Get and print the results for lines connected to the transformers FT2 and T3 
# after adding the new load
after_lines_ft2 = net.res_line.loc[lines_to_ft2.index]
after_lines_t3 = net.res_line.loc[lines_to_t3.index]

print("Results for lines at FT2:\n", after_lines_ft2)
print("Results for lines at T3:\n", after_lines_t3)

Results for lines at FT2:
      p_from_mw  q_from_mvar  p_to_mw  q_to_mvar  pl_mw  ql_mvar  i_from_ka  \
343     0.9242       0.1845  -0.9242    -0.1851 0.0000  -0.0006     0.0192   

     i_to_ka   i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
343   0.0192 0.0192      0.8590        -19.0755    0.8590      -19.0765   

     loading_percent  
343              NaN  
Results for lines at T3:
      p_from_mw  q_from_mvar  p_to_mw  q_to_mvar  pl_mw  ql_mvar  i_from_ka  \
358     0.9242       0.1845  -0.9242    -0.1851 0.0000  -0.0006     0.0192   

     i_to_ka   i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
358   0.0192 0.0192      0.8590        -19.0755    0.8590      -19.0765   

     loading_percent  
358              NaN  


#### Comparison of the results before and after connecting the 1MW load to BARW52 busbar.

In [249]:
barw52_comparison = pd.DataFrame({
    "vm_pu_before": [before_BARW52["vm_pu"]],
    "vm_pu_after": [after_BARW52["vm_pu"]]
}, index=["BARW52"])

print("BARW52 voltage comparison (vm_pu):\n", barw52_comparison)

BARW52 voltage comparison (vm_pu):
         vm_pu_before  vm_pu_after
BARW52        0.9038       0.8960


In [250]:
trafo_comparison = pd.DataFrame({
    "loading_percent_before": [before_ft2["loading_percent"], before_t3["loading_percent"]],
    "loading_percent_after": [after_ft2["loading_percent"], after_t3["loading_percent"]]
}, index=["FT2", "T3"])

print("Trafo loading comparison (%):\n", trafo_comparison)

Trafo loading comparison (%):
      loading_percent_before  loading_percent_after
FT2                  3.9605                 6.0957
T3                   3.9605                 6.0957


In [251]:
line_comparison = pd.DataFrame({
    "loading_percent_before": [before_lines_ft2["loading_percent"].iloc[0], before_lines_t3["loading_percent"].iloc[0]],
    "loading_percent_after": [after_lines_ft2["loading_percent"].iloc[0], after_lines_t3["loading_percent"].iloc[0]],
    "p_from_mw_before": [before_lines_ft2["p_from_mw"].iloc[0], before_lines_t3["p_from_mw"].iloc[0]],
    "p_from_mw_after": [after_lines_ft2["p_from_mw"].iloc[0], after_lines_t3["p_from_mw"].iloc[0]],
    "q_from_mvar_before": [before_lines_ft2["q_from_mvar"].iloc[0], before_lines_t3["q_from_mvar"].iloc[0]],
    "q_from_mvar_after": [after_lines_ft2["q_from_mvar"].iloc[0], after_lines_t3["q_from_mvar"].iloc[0]]
}, index=["Line_FT2", "Line_T3"])

print("Line comparison:\n", line_comparison)

Line comparison:
           loading_percent_before  loading_percent_after  p_from_mw_before  \
Line_FT2                     NaN                    NaN            0.5906   
Line_T3                      NaN                    NaN            0.5906   

          p_from_mw_after  q_from_mvar_before  q_from_mvar_after  
Line_FT2           0.9242              0.1797             0.1845  
Line_T3            0.9242              0.1797             0.1845  


## Add a new generator
A new static generator can be easily created with the *create_sgen" function of pandapower. It requires defining the bus to which the static generator will be connected and its active and reactive power.

In [252]:
# Create a new sgen at the desired bus
sgen_bus = BARW52_index

if sgen_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=sgen_bus)
sgen_p = 0.3
sgen_q = 0
pp.create_sgen(net, bus=sgen_bus, p_mw=sgen_p, q_mvar=sgen_q)
net.sgen.tail(1)

,name,bus,p_mw,q_mvar,min_q_mvar,max_q_mvar,sn_mva,scaling,controllable,id_q_capability_characteristic,...,vn_kv,rdss_ohm,xdss_pu,lrc_pu,RegulatingControl.targetValue,referencePriority,max_p_mw,min_p_mw,RegulatingControl.enabled,vn_kv_bus
8,None,8287,0.3000,0.0000,NaN,NaN,NaN,1.0000,False,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN


## Run a headroom capacity analysis
It is possible to run a hosting capacity study and understand how much load or generation can be connected to a particular node, by incrementing continuously the power (of the load or generator) till when the boundaries of interest are not exceeded.

In this example, for simplicity, we will investigate how much load can be added to the desired bus before exceeding the loading capacity of the grid transformers.

In [253]:
if license_area == "LPN":
    hosting_bus = 18
elif license_area == "SPN":
    hosting_bus = 36
elif license_area == "EPN":
    hosting_bus = 20
else:
    hosting_bus = 0             # bus selected for the analysis
            # bus selected for the analysis
incremental_p_mw = 1        # incremental value of power
if hosting_bus not in net.bus.index:
    print("The given bus does not exist")
    hosting_bus = net.bus.index[0]   # replace the bus with the first bus in the grid
load_index = pp.create_load(net, bus=hosting_bus, p_mw=0, q_mvar=0)
within_hosting_limit = True    # boolean telling if we are still within inside the allowed boundary

# Hosting capacity logic
while within_hosting_limit:
    net.load.loc[load_index, "p_mw"] += incremental_p_mw
    pp.runpp(net, max_iteration=50)
    if np.any(net.res_trafo.loading_percent > 120) or net.trafo.index.size == 0:
        within_hosting_limit = False
        net.load.loc[load_index, "p_mw"] -= incremental_p_mw

# Visualize maximum load that can be connected at the selected bus
display("Maximum load that can be connected at bus " + str(hosting_bus) + " is " + str(net.load.loc[load_index, "p_mw"]) + " MW")

'Maximum load that can be connected at bus 18 is 58.0 MW'